# Statistical and Machine Learning - HW1: Introduction and Learnability
**Student:** Younes Hadli
**Master 2 - Informatique et Mobilité**
**University of Haute-Alsace (FST)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

## 1. Data Generation
We consider the input space $\mathcal{X} = [0, 1]$ and a target threshold classifier $f_\theta(x) = 1 \text{ if } x > \theta \text{ else } 0$.

In [ ]:
class ThresholdClassifier:
    def __init__(self, theta):
        self.theta = theta
        
    def predict(self, X):
        return (X > self.theta).astype(int)

def generate_data(m, theta, noise_rate=0.0, agnostic=False):
    X = np.random.uniform(0, 1, m)
    y = (X > theta).astype(int)
    if agnostic and noise_rate > 0.0:
        flip_mask = np.random.rand(m) < noise_rate
        y[flip_mask] = 1 - y[flip_mask]
    return X, y

# Visualize at least one generated dataset
theta_true = 0.5
X_sample, y_sample = generate_data(m=50, theta=theta_true)

plt.figure(figsize=(9, 2.5))
plt.scatter(X_sample[y_sample == 0], np.zeros_like(X_sample[y_sample == 0]), color='blue', label='Class 0', s=50, marker='o')
plt.scatter(X_sample[y_sample == 1], np.zeros_like(X_sample[y_sample == 1]), color='red', label='Class 1', s=50, marker='x')
plt.axvline(x=theta_true, color='green', linestyle='--', label=f'True Target $\theta$ = {theta_true}')
plt.yticks([])
plt.xlabel('Input Space $\mathcal{X} = [0, 1]$')
plt.title('Visualization of Generated Training Dataset (Realizable)')
plt.legend(loc='upper right')
plt.show()

## 2. Implementation of Empirical Risk Minimization (ERM)
We implement an ERM algorithm over the hypothesis class of threshold classifiers by testing candidate thresholds between sorted data points.

In [ ]:
def train_erm(X, y):
    sorted_indices = np.argsort(X)
    X_sorted = X[sorted_indices]
    y_sorted = y[sorted_indices]
    
    best_theta = 0.0
    min_empirical_error = float('inf')
    
    candidate_thetas = [0.0] + [(X_sorted[i] + X_sorted[i+1]) / 2 for i in range(len(X_sorted) - 1)] + [1.0]
    
    for theta in candidate_thetas:
        clf = ThresholdClassifier(theta)
        preds = clf.predict(X)
        err = np.mean(preds != y)
        if err < min_empirical_error:
            min_empirical_error = err
            best_theta = theta
            
    return ThresholdClassifier(best_theta), min_empirical_error

## 3. Performance Evaluation
For various sample sizes, we compute both empirical and true errors.

In [ ]:
sample_sizes = [5, 10, 20, 50, 100, 200]
X_test = np.random.uniform(0, 1, 5000)
y_test = (X_test > theta_true).astype(int)

emp_errors = []
true_errors = []

for m in sample_sizes:
    X_train, y_train = generate_data(m, theta_true)
    clf, emp_err = train_erm(X_train, y_train)
    y_pred_test = clf.predict(X_test)
    true_err = np.mean(y_pred_test != y_test)
    
    emp_errors.append(emp_err)
    true_errors.append(true_err)

plt.figure(figsize=(8, 4))
plt.plot(sample_sizes, emp_errors, marker='o', label='Empirical Error ($L_S$)')
plt.plot(sample_sizes, true_errors, marker='s', label='True Error ($L_D$)')
plt.xlabel('Training Sample Size ($m$)')
plt.ylabel('Error Rate')
plt.title('Empirical Error vs True Error across Sample Sizes')
plt.legend()
plt.grid(True)
plt.show()

## 4. Experimental Verification of PAC Learning
We repeat the experiment multiple times to estimate $Pr(L(h) > \epsilon)$ as $m$ increases.

In [ ]:
n_repetitions = 500
epsilon = 0.15
delta_threshold = 0.05
pac_probabilities = []

for m in sample_sizes:
    exceeded_count = 0
    for _ in range(n_repetitions):
        X_train, y_train = generate_data(m, theta_true, agnostic=False)
        clf, _ = train_erm(X_train, y_train)
        y_pred_test = clf.predict(X_test)
        true_err = np.mean(y_pred_test != y_test)
        if true_err > epsilon:
            exceeded_count += 1
    prob = exceeded_count / n_repetitions
    pac_probabilities.append(prob)

plt.figure(figsize=(8, 4))
plt.plot(sample_sizes, pac_probabilities, marker='o', color='purple', label='Estimated $Pr(L(h) > \epsilon)$')
plt.axhline(y=delta_threshold, color='red', linestyle='--', label='Failure Tolerance $\delta$')
plt.xlabel('Training Sample Size ($m$)')
plt.ylabel('Probability')
plt.title('PAC Learning Verification: Probability of Exceeding $\epsilon$')
plt.legend()
plt.grid(True)
plt.show()

## 5. From PAC to Agnostic PAC (Introducing Label Noise)

In [ ]:
noise_rate = 0.15
agnostic_probabilities = []

for m in sample_sizes:
    exceeded_count = 0
    for _ in range(n_repetitions):
        X_train, y_train = generate_data(m, theta_true, noise_rate=noise_rate, agnostic=True)
        clf, _ = train_erm(X_train, y_train)
        y_pred_test = clf.predict(X_test)
        true_err = np.mean(y_pred_test != y_test)
        if true_err > (epsilon + noise_rate / 2):
            exceeded_count += 1
    prob = exceeded_count / n_repetitions
    agnostic_probabilities.append(prob)

plt.figure(figsize=(8, 4))
plt.plot(sample_sizes, pac_probabilities, marker='o', label='PAC (Realizable)')
plt.plot(sample_sizes, agnostic_probabilities, marker='s', label='Agnostic PAC (Noisy Labels)')
plt.xlabel('Training Sample Size ($m$)')
plt.ylabel('Probability Exceeding Error Bound')
plt.title('Comparison between PAC and Agnostic PAC Settings')
plt.legend()
plt.grid(True)
plt.show()

## 6. Conclusion: PAC vs Agnostic PAC

Based on the experimental observations:
1. **Empirical Error:** In the realizable setting, the empirical error reaches zero as the model perfectly fits clean data. In the agnostic setting, due to independent label noise, the empirical error cannot drop to zero.
2. **Generalization Performance:** Increasing the training sample size ($m$) improves generalization across both frameworks, reducing the probability of high generalization error.
3. **Framework Comparison:** The PAC framework guarantees near-zero true error under realizability, whereas the Agnostic PAC framework gracefully adapts to noisy environments by bounding performance relative to the best hypothesis within $\mathcal{H}$.